# Step 4 — Multi-Sensor Track Fusion (Upgraded) ✅

| | |
|---|---|
| **Input** | `output/step_3/lidar/track_*.json`, `output/step_3/radar/track_*.json` (tuple format) |
| | `output/step_3/camera/track_*.json` (dict format, from Step 3.3) |
| **Outputs** | `output/step_4/fused_tracks_all.csv` — one row per fused-track point (fast, primary output) |
| | `output/step_4/fused/track_<id>.json` — one file per fused track (now feasible — thousands, not millions) |
| | `output/step_4/fusion_summary.csv` |
| **Used by** | Step 5 (TTC estimation) |

---

### The real cause of the 22-hour runtime

`active_fused_objects = detections` at the end of each loop carried forward **every raw detection** from the current frame (~10,000+ points, dominated by radar) as the matching pool for the next frame. The nested loop then ran roughly 100 million scalar `euclidean()` calls per frame. That's the 196.72s/iteration you measured — not primarily the file writes, though writing 1M+ individual JSON files made it worse.

### The fix: fuse at the TRACK level, not the raw-point level

Steps 3.1/3.2/3.3 already turned millions of raw points into a few thousand clean, deduplicated tracks. This notebook now fuses those tracks — a problem that's orders of magnitude smaller and doesn't need any raw point data at all.

### Also fixed
- **Format compatibility** — loaders now correctly parse Step 3.1/3.2's tuple format and Step 3.3's dict/trajectory format.
- **Converted from Colab (Drive mount + zip) to local, config.py-based** — consistent with the rest of the pipeline.
- **Proper one-shot Hungarian assignment per frame** instead of nested nearest-neighbor nested loops.
- **Within-frame sensor merging** — if LiDAR and radar both observe the same real object in the same frame, they're merged into one observation (averaged position, sensors recorded) before being matched against existing fused tracks. This is the actual "fusion" step that was largely absent before.
- **Track eviction** — same `MAX_MISSED_FRAMES` pattern as Steps 3.1–3.3.

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

import shutil
from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import STEP0_DIR, STEP3_DIR, STEP4_DIR

LIDAR_TRACKS_DIR  = STEP3_DIR / "lidar"
RADAR_TRACKS_DIR  = STEP3_DIR / "radar"
CAMERA_TRACKS_DIR = STEP3_DIR / "camera"
FUSED_OUT_DIR      = STEP4_DIR / "fused"
if FUSED_OUT_DIR.exists():
    shutil.rmtree(FUSED_OUT_DIR)   # FIXED: clear stale per-object files before this run (was: accumulated silently across re-runs, never cleared)
FUSED_OUT_DIR.mkdir(parents=True, exist_ok=True)

for p, name in [(LIDAR_TRACKS_DIR, "Step 3.1"), (RADAR_TRACKS_DIR, "Step 3.2"), (CAMERA_TRACKS_DIR, "Step 3.3")]:
    if not p.exists():
        raise FileNotFoundError(f"{name} output not found at {p} — run that step first.")

print(f"✅ LIDAR_TRACKS_DIR : {LIDAR_TRACKS_DIR}")
print(f"✅ RADAR_TRACKS_DIR : {RADAR_TRACKS_DIR}")
print(f"✅ CAMERA_TRACKS_DIR: {CAMERA_TRACKS_DIR}")
print(f"✅ FUSED_OUT_DIR    : {FUSED_OUT_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output


✅ LIDAR_TRACKS_DIR : F:\Sensor fusion Research\output\step_3\lidar
✅ RADAR_TRACKS_DIR : F:\Sensor fusion Research\output\step_3\radar
✅ CAMERA_TRACKS_DIR: F:\Sensor fusion Research\output\step_3\camera
✅ FUSED_OUT_DIR    : F:\Sensor fusion Research\output\step_4\fused


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Named constants
# ─────────────────────────────────────────────────────────────────

INTRA_FRAME_MERGE_THRESH = 2.5   # metres — merge same-frame observations from different sensors this close together
FUSION_DIST_THRESHOLD    = 3.0   # metres — max distance to match a frame observation to an existing fused track
MAX_MISSED_FRAMES        = 3     # frames before a fused track is evicted

print(f"✅ INTRA_FRAME_MERGE_THRESH = {INTRA_FRAME_MERGE_THRESH}m")
print(f"✅ FUSION_DIST_THRESHOLD    = {FUSION_DIST_THRESHOLD}m")
print(f"✅ MAX_MISSED_FRAMES        = {MAX_MISSED_FRAMES}")

✅ INTRA_FRAME_MERGE_THRESH = 2.5m
✅ FUSION_DIST_THRESHOLD    = 3.0m
✅ MAX_MISSED_FRAMES        = 3


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Loaders: convert each sensor's track format into a common
# per-sample observation list: {sensor, source_track_id, pos}, and also
# keep each source track's own point sequence (used by the next cell to
# run independent per-sensor UKF smoothing before any cross-sensor merge).
# ─────────────────────────────────────────────────────────────────

import json
from collections import defaultdict

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)


def load_lidar_or_radar_tracks(track_dir, sensor_name):
    """Step 3.1/3.2 format: each track_*.json is a list of
    (sample_id, timestamp, [x,y,z]) tuples."""
    observations_by_sample = defaultdict(list)
    tracks_by_id = {}
    for file in track_dir.glob("track_*.json"):
        source_track_id = file.stem
        with open(file) as f:
            points = json.load(f)
        points = sorted(points, key=lambda p: p[1])
        tracks_by_id[source_track_id] = points
        for sample_id, timestamp, pos in points:
            observations_by_sample[sample_id].append({
                "sensor": sensor_name,
                "source_track_id": source_track_id,
                "pos": pos
            })
    return observations_by_sample, tracks_by_id


def load_camera_tracks(track_dir):
    """Step 3.3 format: each track_*.json is a dict with a 'trajectory' list.
    Trajectory points carry no timestamp of their own -- resolved via
    samples_index, same as Step 5's loader does."""
    observations_by_sample = defaultdict(list)
    tracks_by_id = {}
    for file in track_dir.glob("track_*.json"):
        with open(file) as f:
            data = json.load(f)
        source_track_id = data["track_id"]
        points = []
        for pt in data["trajectory"]:
            ts = samples_index.get(pt["sample_id"], {}).get("timestamp_us")
            points.append((pt["sample_id"], ts, pt["pos"]))
            observations_by_sample[pt["sample_id"]].append({
                "sensor": "camera",
                "source_track_id": source_track_id,
                "pos": pt["pos"]
            })
        points = sorted(points, key=lambda p: (p[1] is None, p[1]))
        tracks_by_id[source_track_id] = points
    return observations_by_sample, tracks_by_id


lidar_by_sample,  lidar_tracks_by_id  = load_lidar_or_radar_tracks(LIDAR_TRACKS_DIR, "lidar")
radar_by_sample,  radar_tracks_by_id  = load_lidar_or_radar_tracks(RADAR_TRACKS_DIR, "radar")
camera_by_sample, camera_tracks_by_id = load_camera_tracks(CAMERA_TRACKS_DIR)

# Diagnostic: how many detections is each sensor actually contributing per frame on average?
for name, by_sample in [("lidar", lidar_by_sample), ("radar", radar_by_sample), ("camera", camera_by_sample)]:
    total_points = sum(len(v) for v in by_sample.values())
    n_samples_with_data = sum(1 for v in by_sample.values() if len(v) > 0)
    avg_per_frame = total_points / n_samples_with_data if n_samples_with_data > 0 else 0
    print(f"{name}: {total_points} total points, {n_samples_with_data} samples with data, "
          f"avg {avg_per_frame:.1f} detections/frame")

n_lidar  = sum(len(v) for v in lidar_by_sample.values())
n_radar  = sum(len(v) for v in radar_by_sample.values())
n_camera = sum(len(v) for v in camera_by_sample.values())

print(f"✅ Loaded {n_lidar} LiDAR track-points")
print(f"✅ Loaded {n_radar} Radar track-points")
print(f"✅ Loaded {n_camera} Camera track-points")
print(f"   Total: {n_lidar + n_radar + n_camera} (compare to the original's millions — this is tracks, not raw points)")

lidar: 16179 total points, 404 samples with data, avg 40.0 detections/frame
radar: 5490 total points, 398 samples with data, avg 13.8 detections/frame
camera: 4399 total points, 377 samples with data, avg 11.7 detections/frame
✅ Loaded 16179 LiDAR track-points
✅ Loaded 5490 Radar track-points
✅ Loaded 4399 Camera track-points
   Total: 26068 (compare to the original's millions — this is tracks, not raw points)


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 3b — Per-sensor UKF/CTRV state smoothing (state-level fusion, part 1)
#
# Each sensor's own track is filtered independently and continuously with
# the SAME UKF/CTRV model Step 5 uses for TTC, BEFORE any cross-sensor
# merging happens. The cross-sensor merge below then combines each
# sensor's already-smoothed, temporally-consistent (x, y, vx, vy) state
# instead of raw single-frame positions -- so a track's velocity keeps
# each contributing sensor's own filtering history instead of being
# re-derived from scratch on a merged stream that resets every time the
# sensor composition changes frame-to-frame.
# ─────────────────────────────────────────────────────────────────

import numpy as np


def merwe_sigma_points(x, P, alpha=1e-3, beta=2.0, kappa=0.0):
    n = x.size
    lam = alpha**2 * (n + kappa) - n
    c = n + lam
    U = np.linalg.cholesky(P * c)
    Wm = np.full(2*n+1, 1/(2*c)); Wc = np.full(2*n+1, 1/(2*c))
    Wm[0] = lam / c; Wc[0] = lam / c + (1 - alpha**2 + beta)
    sigmas = np.zeros((2*n+1, n))
    sigmas[0] = x
    for i in range(n):
        sigmas[i+1]   = x + U[:, i]
        sigmas[n+i+1] = x - U[:, i]
    return sigmas, Wm, Wc


def ctrv_process(sigma, dt):
    x, y, v, yaw, yawd = sigma
    if abs(yawd) > 1e-6:
        x_p = x + v/yawd * (np.sin(yaw + yawd*dt) - np.sin(yaw))
        y_p = y + v/yawd * (-np.cos(yaw + yawd*dt) + np.cos(yaw))
    else:
        x_p = x + v * np.cos(yaw) * dt
        y_p = y + v * np.sin(yaw) * dt
    return np.array([x_p, y_p, v, yaw + yawd*dt, yawd])


def ukf_predict(x, P, Q, dt):
    sigmas, Wm, Wc = merwe_sigma_points(x, P)
    sigmas_f = np.array([ctrv_process(s, dt) for s in sigmas])
    x_pred = np.sum(Wm[:, None] * sigmas_f, axis=0)
    P_pred = Q.copy()
    for i in range(sigmas_f.shape[0]):
        y = sigmas_f[i] - x_pred
        y[3] = (y[3] + np.pi) % (2*np.pi) - np.pi
        P_pred += Wc[i] * np.outer(y, y)
    return x_pred, P_pred, sigmas_f, Wm, Wc


def ukf_update(x_pred, P_pred, sigmas_f, Wm, Wc, z, R):
    Z = np.array([s[:2] for s in sigmas_f])
    z_pred = np.sum(Wm[:, None] * Z, axis=0)
    S = R.copy()
    for i in range(Z.shape[0]):
        dz = Z[i] - z_pred
        S += Wc[i] * np.outer(dz, dz)
    n = x_pred.size
    Tc = np.zeros((n, 2))
    for i in range(Z.shape[0]):
        dx = sigmas_f[i] - x_pred
        dx[3] = (dx[3] + np.pi) % (2*np.pi) - np.pi
        dz = Z[i] - z_pred
        Tc += Wc[i] * np.outer(dx, dz)
    K = Tc @ np.linalg.inv(S)
    dz = z - z_pred
    x_upd = x_pred + K @ dz
    P_upd = P_pred - K @ S @ K.T
    return x_upd, P_upd


def smooth_track_states(points):
    """points: list of (sample_id, timestamp, pos) for ONE source track,
    already sorted by timestamp. Returns {sample_id: (x, y, vx, vy, z)}
    using the same UKF/CTRV model as Step 5 for 3+ points, direct finite
    difference for exactly 2, or zero velocity for a single detection."""
    pts = [p for p in points if p[1] is not None]
    n = len(pts)
    if n == 0:
        return {}
    if n == 1:
        sid, ts, pos = pts[0]
        z = pos[2] if len(pos) > 2 else 0.0
        return {sid: (pos[0], pos[1], 0.0, 0.0, z)}
    if n == 2:
        (s0, t0, p0), (s1, t1, p1) = pts
        dt = (t1 - t0) / 1e6
        vx, vy = (0.0, 0.0) if dt <= 0 else ((p1[0]-p0[0])/dt, (p1[1]-p0[1])/dt)
        z0 = p0[2] if len(p0) > 2 else 0.0
        z1 = p1[2] if len(p1) > 2 else 0.0
        return {s0: (p0[0], p0[1], vx, vy, z0), s1: (p1[0], p1[1], vx, vy, z1)}

    dt0 = (pts[1][1] - pts[0][1]) / 1e6
    if dt0 <= 0:
        dt0 = 1e-3
    dx = (pts[1][2][0] - pts[0][2][0]) / dt0
    dy = (pts[1][2][1] - pts[0][2][1]) / dt0
    v0 = np.hypot(dx, dy)
    yaw0 = np.arctan2(dy, dx) if v0 > 1e-6 else 0.0

    x = np.array([pts[0][2][0], pts[0][2][1], v0, yaw0, 0.0])
    P = np.diag([1.0, 1.0, 5.0, 0.5, 0.5])
    Q = np.diag([0.2]*5)
    R = np.diag([0.5, 0.5])   # one sensor's own track: homogeneous source, fixed R is fine here

    out = {}
    prev_ts = pts[0][1]
    for sid, ts, pos in pts:
        dt = max((ts - prev_ts) / 1e6, 1e-3)
        prev_ts = ts
        z_meas = np.array([pos[0], pos[1]])
        x_pred, P_pred, sigmas_f, Wm, Wc = ukf_predict(x, P, Q, dt)
        x, P = ukf_update(x_pred, P_pred, sigmas_f, Wm, Wc, z_meas, R)
        v, yaw = x[2], x[3]
        vx, vy = v * np.cos(yaw), v * np.sin(yaw)
        z_raw = pos[2] if len(pos) > 2 else 0.0
        out[sid] = (x[0], x[1], vx, vy, z_raw)
    return out


smoothed_state_lookup = {}
for sensor_name, tracks_by_id in [("lidar", lidar_tracks_by_id), ("radar", radar_tracks_by_id), ("camera", camera_tracks_by_id)]:
    for source_track_id, points in tracks_by_id.items():
        for sample_id, state in smooth_track_states(points).items():
            smoothed_state_lookup[(sensor_name, source_track_id, sample_id)] = state

smoothed_observations_by_sample = defaultdict(list)
for (sensor_name, source_track_id, sample_id), (sx, sy, svx, svy, sz) in smoothed_state_lookup.items():
    smoothed_observations_by_sample[sample_id].append({
        "sensor": sensor_name, "source_track_id": source_track_id,
        "x": sx, "y": sy, "vx": svx, "vy": svy, "z": sz
    })

print(f"✅ Smoothed {len(smoothed_state_lookup)} (sensor, track, sample) states independently per source track.")

✅ Smoothed 26068 (sensor, track, sample) states independently per source track.


In [5]:
# CELL 4 - Within-frame sensor merging (the actual "fusion" step)
#
# FIXED (Bug A): only merge observations from DIFFERENT sensors - two
# LiDAR (or two radar, or two camera) detections never get merged with
# each other, even if they are close. Two real pedestrians standing near
# each other, both seen only by LiDAR, must stay two separate detections.
#
# FIXED (Bug B): the old pass was greedy, not a real union-find - a chain
# of 3+ pairwise-close detections (A close to B, B close to C, A far from
# C) only grouped the first pair and left the third alone. This uses a
# proper disjoint-set union so the whole chain ends up in one cluster.
#
# STATE-LEVEL FUSION: this now merges each sensor's already-UKF-smoothed
# (x, y, vx, vy) state from the cell above, not raw single-frame
# positions -- weighted by each sensor's approximate accuracy
# (SENSOR_TRUST_WEIGHT), so a fused track's velocity keeps each
# contributing sensor's own temporal filtering instead of being
# re-derived from scratch on a merged position stream in Step 5.

# Inverse-variance-style trust weights from each sensor's approximate
# positional accuracy (LiDAR ~0.15m, radar sloppier laterally ~0.5m,
# camera-derived range least precise, especially for far objects ~1.0m).
# weight ~ 1 / sigma^2 -- the more accurate sensor dominates a merge.
SENSOR_TRUST_WEIGHT = {
    "lidar": 44.0,
    "radar": 4.0,
    "camera": 1.0,
}


def merge_frame_observations(observations, merge_thresh):
    """Union-find clustering over already-smoothed per-sensor states. Fine
    at this scale (tens of observations per frame, not thousands), so an
    O(n^2) edge pass is fast."""
    n = len(observations)
    if n == 0:
        return []

    positions = np.array([[o["x"], o["y"]] for o in observations])
    sensors = [o["sensor"] for o in observations]
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            if sensors[i] == sensors[j]:
                continue  # never merge two detections from the same sensor
            if np.linalg.norm(positions[i] - positions[j]) < merge_thresh:
                union(i, j)

    clusters = {}
    for i in range(n):
        clusters.setdefault(find(i), []).append(i)

    merged = []
    for member_idx in clusters.values():
        members = [observations[i] for i in member_idx]
        weights = np.array([SENSOR_TRUST_WEIGHT.get(m["sensor"], 1.0) for m in members])
        state = np.array([[m["x"], m["y"], m["vx"], m["vy"], m["z"]] for m in members])
        avg = (weights[:, None] * state).sum(axis=0) / weights.sum()
        merged.append({
            "pos": [avg[0], avg[1], avg[4]],
            "vx": avg[2], "vy": avg[3],
            "sensors": sorted({m["sensor"] for m in members}),
            "source_track_ids": {m["sensor"]: m["source_track_id"] for m in members}
        })
    return merged


print("✅ merge_frame_observations() defined — cross-sensor only, union-find, trust-weighted, over pre-smoothed states.")

✅ merge_frame_observations() defined — cross-sensor only, union-find, trust-weighted, over pre-smoothed states.


In [6]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Fused tracker: one Hungarian assignment per frame + eviction
# ─────────────────────────────────────────────────────────────────

import uuid
from scipy.optimize import linear_sum_assignment


class FusedTracker:
    def __init__(self, dist_thresh, max_missed):
        self.active_tracks = {}
        self.finished_tracks = {}
        self.dist_thresh = dist_thresh
        self.max_missed = max_missed

    def update(self, merged_observations, sample_id, timestamp):
        track_ids = list(self.active_tracks.keys())
        n_tracks, n_obs = len(track_ids), len(merged_observations)

        matched_track_idx, matched_obs_idx = set(), set()

        if n_tracks > 0 and n_obs > 0:
            # cost = np.zeros((n_tracks, n_obs))
            # for i, tid in enumerate(track_ids):
            #     last_pos = np.array(self.active_tracks[tid]["points"][-1]["pos"])
            #     for j, obs in enumerate(merged_observations):
            #         cost[i, j] = np.linalg.norm(np.array(obs["pos"]) - last_pos)

            cost = np.zeros((n_tracks, n_obs))
            for i, tid in enumerate(track_ids):
                pts = self.active_tracks[tid]["points"]       # [{"timestamp":..., "pos":[...]}, ...]
                last_pos = np.array(pts[-1]["pos"], dtype=float)
                pred = last_pos
                if len(pts) >= 2 and pts[-1]["timestamp"] is not None and pts[-2]["timestamp"] is not None:
                    dt_prev = (pts[-1]["timestamp"] - pts[-2]["timestamp"]) / 1e6
                    dt_now  = (timestamp              - pts[-1]["timestamp"]) / 1e6
                    if dt_prev > 0 and dt_now > 0:
                        vel = (last_pos - np.array(pts[-2]["pos"], dtype=float)) / dt_prev
                        spd = np.linalg.norm(vel)
                        if spd > 30.0:
                            vel = vel / spd * 30.0
                        pred = last_pos + vel * dt_now
                for j, obs in enumerate(merged_observations):
                    cost[i, j] = np.linalg.norm(np.array(obs["pos"], dtype=float) - pred)
            
            row_idx, col_idx = linear_sum_assignment(cost)
            for r, c in zip(row_idx, col_idx):
                if cost[r, c] < self.dist_thresh:
                    tid = track_ids[r]
                    obs = merged_observations[c]
                    self.active_tracks[tid]["points"].append({
                        "sample_id": sample_id, "timestamp": timestamp,
                        "pos": obs["pos"], "vx": obs["vx"], "vy": obs["vy"],
                        "sensors": obs["sensors"]
                    })
                    self.active_tracks[tid]["missed"] = 0
                    matched_track_idx.add(r)
                    matched_obs_idx.add(c)

        for i, tid in enumerate(track_ids):
            if i not in matched_track_idx:
                self.active_tracks[tid]["missed"] += 1
                if self.active_tracks[tid]["missed"] > self.max_missed:
                    self.finished_tracks[tid] = self.active_tracks.pop(tid)

        for j in range(n_obs):
            if j not in matched_obs_idx:
                obs = merged_observations[j]
                tid = f"fused_{uuid.uuid4().hex[:8]}"
                self.active_tracks[tid] = {
                    "points": [{
                        "sample_id": sample_id, "timestamp": timestamp,
                        "pos": obs["pos"], "vx": obs["vx"], "vy": obs["vy"],
                        "sensors": obs["sensors"]
                    }],
                    "missed": 0
                }

    def all_tracks(self):
        return {**self.finished_tracks, **self.active_tracks}


print("✅ FusedTracker defined.")

✅ FusedTracker defined.


In [7]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 — Main fusion loop
# ─────────────────────────────────────────────────────────────────

from tqdm import tqdm
import time

all_sample_ids = sorted(
    set(lidar_by_sample.keys()) | set(radar_by_sample.keys()) | set(camera_by_sample.keys())
)

tracker = FusedTracker(dist_thresh=FUSION_DIST_THRESHOLD, max_missed=MAX_MISSED_FRAMES)

start_time = time.time()
for sample_id in tqdm(all_sample_ids, desc="Fusing tracks"):
    frame_observations = smoothed_observations_by_sample.get(sample_id, [])

    merged = merge_frame_observations(frame_observations, INTRA_FRAME_MERGE_THRESH)

    timestamp = samples_index.get(sample_id, {}).get("timestamp_us", None)
    tracker.update(merged, sample_id, timestamp)

elapsed = time.time() - start_time
all_tracks = tracker.all_tracks()

print(f"\n✅ Fusion complete in {elapsed:.1f} seconds (was 22 hours, 4 minutes originally).")
print(f"   Total fused tracks: {len(all_tracks)}")

Fusing tracks:   0%|          | 0/404 [00:00<?, ?it/s]

Fusing tracks:   2%|▏         | 8/404 [00:00<00:05, 78.81it/s]

Fusing tracks:   4%|▍         | 16/404 [00:00<00:06, 57.81it/s]

Fusing tracks:   6%|▌         | 23/404 [00:00<00:07, 51.32it/s]

Fusing tracks:   7%|▋         | 29/404 [00:00<00:07, 51.20it/s]

...

Fusing tracks:  97%|█████████▋| 391/404 [00:06<00:00, 47.83it/s]

Fusing tracks:  98%|█████████▊| 396/404 [00:06<00:00, 47.68it/s]

Fusing tracks:  99%|█████████▉| 401/404 [00:06<00:00, 47.28it/s]

Fusing tracks: 100%|██████████| 404/404 [00:06<00:00, 63.09it/s]


✅ Fusion complete in 6.4 seconds (was 22 hours, 4 minutes originally).
   Total fused tracks: 4785


In [8]:
# ─────────────────────────────────────────────────────────────────
# CELL 7 — Save: one combined CSV (primary, fast) + per-track JSON files
# (now feasible — thousands of files, not 1,057,292)
# ─────────────────────────────────────────────────────────────────

import pandas as pd

csv_rows = []
n_saved_json = 0

for tid, track in all_tracks.items():
    if len(track["points"]) < 2:
        continue

    for pt in track["points"]:
        csv_rows.append({
            "fused_id": tid,
            "sample_id": pt["sample_id"],
            "timestamp": pt["timestamp"],
            "x": pt["pos"][0], "y": pt["pos"][1], "z": pt["pos"][2] if len(pt["pos"]) > 2 else 0.0,
            "vx": pt.get("vx", 0.0), "vy": pt.get("vy", 0.0),
            "sensors": "+".join(pt["sensors"])
        })

    with open(FUSED_OUT_DIR / f"track_{tid}.json", "w") as f:
        json.dump(track["points"], f, indent=2)
    n_saved_json += 1

fused_df = pd.DataFrame(csv_rows)
csv_path = STEP4_DIR / "fused_tracks_all.csv"
fused_df.to_csv(csv_path, index=False)

print(f"✅ Combined CSV saved: {csv_path} ({len(fused_df)} rows)")
print(f"✅ {n_saved_json} individual track JSON files saved to: {FUSED_OUT_DIR}")

✅ Combined CSV saved: F:\Sensor fusion Research\output\step_4\fused_tracks_all.csv (19263 rows)
✅ 4138 individual track JSON files saved to: F:\Sensor fusion Research\output\step_4\fused


In [9]:
# ─────────────────────────────────────────────────────────────────
# CELL 8 — Summary: sensor contribution breakdown
# ─────────────────────────────────────────────────────────────────

multi_sensor_points = fused_df[fused_df["sensors"].str.contains(r"\+")]
single_sensor_points = fused_df[~fused_df["sensors"].str.contains(r"\+")]

print(f"Total fused track points     : {len(fused_df)}")
print(f"Points from 2+ sensors merged: {len(multi_sensor_points)} "
      f"({len(multi_sensor_points)/len(fused_df)*100:.1f}%)")
print(f"Points from a single sensor  : {len(single_sensor_points)}")

print("\nSensor combination breakdown:")
display(fused_df["sensors"].value_counts().head(10))

track_lengths = fused_df.groupby("fused_id").size()
summary_path = STEP4_DIR / "fusion_summary.csv"
track_lengths.reset_index(name="track_length").to_csv(summary_path, index=False)

print(f"\nMean fused track length: {track_lengths.mean():.1f} frames")
print(f"📄 Summary saved: {summary_path}")

Total fused track points     : 19263
Points from 2+ sensors merged: 2754 (14.3%)
Points from a single sensor  : 16509

Sensor combination breakdown:


sensors
lidar                 11802
radar                  3472
camera+lidar           1417
camera                 1235
lidar+radar             798
camera+lidar+radar      373
camera+radar            166
Name: count, dtype: int64


Mean fused track length: 4.7 frames
📄 Summary saved: F:\Sensor fusion Research\output\step_4\fusion_summary.csv


In [10]:
# Experiment: does loosening INTRA_FRAME_MERGE_THRESH increase multi-sensor merge rate
# without hurting merged-group accuracy?

def run_fusion_with_threshold(merge_thresh, dist_thresh=FUSION_DIST_THRESHOLD, max_missed=MAX_MISSED_FRAMES):
    tracker = FusedTracker(dist_thresh=dist_thresh, max_missed=max_missed)
    for sample_id in all_sample_ids:
        frame_observations = smoothed_observations_by_sample.get(sample_id, [])
        merged = merge_frame_observations(frame_observations, merge_thresh)
        timestamp = samples_index.get(sample_id, {}).get("timestamp_us", None)
        tracker.update(merged, sample_id, timestamp)

    all_tracks = tracker.all_tracks()
    multi_sensor_count = sum(
        1 for t in all_tracks.values() for pt in t["points"] if len(pt["sensors"]) > 1
    )
    total_points = sum(len(t["points"]) for t in all_tracks.values())
    merge_rate = multi_sensor_count / total_points * 100 if total_points > 0 else 0
    return len(all_tracks), total_points, merge_rate


print(f"{'Threshold':<10} {'Tracks':<10} {'Points':<10} {'Merge Rate':<12}")
for thresh in [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    n_tracks, n_points, merge_rate = run_fusion_with_threshold(thresh)
    marker = " ← current" if thresh == INTRA_FRAME_MERGE_THRESH else ""
    print(f"{thresh:<10} {n_tracks:<10} {n_points:<10} {merge_rate:<10.1f}%{marker}")

Threshold  Tracks     Points     Merge Rate  


1.5        5061       22114      12.2      %


2.0        4925       20969      13.4      %


2.5        4785       19910      14.1      % ← current


3.0        4636       18891      14.7      %


3.5        4575       17947      15.0      %


4.0        4439       17019      15.2      %
